# Hatch: Modern Python Build System

## The Core Question

You've written a Python package (a CFD solver, a post-processing tool, a utility library). How do you **build** it into a distributable `.whl` file and **install** it so others can `pip install` it?

**Hatch** is a modern Python build system and project manager. It replaces `setuptools` + `pip` + `virtualenv` with a single, fast, well-designed tool.

## Why Hatch?

- **Modern**: uses `pyproject.toml` (PEP 621) — no more `setup.py`
- **Fast**: written with performance in mind, parallel builds
- **Flexible**: supports multiple build backends, custom plugins
- **All-in-one**: build, test, lint, publish — one CLI

## The Workflow

```
hatch new my_cfd_tool
        |
        v
  Edit pyproject.toml (name, version, dependencies)
        |
        v
  hatch build  (creates dist/*.whl + dist/*.tar.gz)
        |
        v
  hatch install dist/my_cfd_tool-0.1.0-py3-none-any.whl
        |
        v
  pip install my_cfd_tool  (or from PyPI)
```

## Setup


In [ ]:
import subprocess
import shutil

def check_hatch():
    try:
        result = subprocess.run(['hatch', '--version'], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            return result.stdout.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return None

hatch_version = check_hatch()
if hatch_version:
    print(f'hatch is installed: {hatch_version}')
    HAS_HATCH = True
else:
    print('hatch is NOT installed.')
    print('Install: pip install hatch')
    print()
    print('This notebook will show the commands and expected output.')
    HAS_HATCH = False

## 1. Create a Project

**Concept**: `hatch new` creates a project with `pyproject.toml`, a source package, and a test file. The structure follows PEP 621 — no `setup.py`, no `setup.cfg`.


In [ ]:
if HAS_HATCH:
    import os
    test_dir = './hatch_demo'
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    
    result = subprocess.run(['hatch', 'new', test_dir], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    
    for root, dirs, files in os.walk(test_dir):
        level = root.replace(test_dir, '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in sorted(files):
            print(f'{indent}  {f}')
else:
    print('Expected output of `hatch new my_cfd_tool`:')
    print()
    print('  my_cfd_tool/')
    print('    pyproject.toml')
    print('    src/')
    print('      my_cfd_tool/')
    print('        __init__.py')
    print('    tests/')
    print('      test_my_cfd_tool.py')
    print()
    print('pyproject.toml:')
    print('  [project]')
    print('  name = "my_cfd_tool"')
    print('  version = "0.0.1"')
    print('  requires-python = ">=3.13"')

## 2. Configure the Package

**Concept**: `pyproject.toml` is the single source of truth. You define the package name, version, dependencies, and build backend here. No more `setup.py` with its quirks.


In [ ]:
if HAS_HATCH:
    with open('./hatch_demo/pyproject.toml', 'r') as f:
        print(f.read())
else:
    print('Example pyproject.toml for a CFD tool:')
    print()
    print('  [project]')
    print('  name = "cfd-post"')
    print('  version = "0.1.0"')
    print('  description = "Post-processing for CFD simulations"')
    print('  requires-python = ">=3.10"')
    print('  dependencies = [')
    print('      "numpy>=1.24",')
    print('      "scipy>=1.10",')
    print('      "matplotlib>=3.7",')
    print('  ]')
    print()
    print('  [project.optional-dependencies]')
    print('  dev = ["pytest", "ruff", "mypy"]')
    print()
    print('  [build-system]')
    print('  requires = ["hatchling"]')
    print('  build-backend = "hatchling.build"')
    print()
    print('  [tool.hatch.build.targets.wheel]')
    print('  packages = ["src/cfd_post"]')

## 3. Build the Package

**Concept**: `hatch build` creates a wheel (`.whl`) and an sdist (`.tar.gz`) in `dist/`. The wheel is what `pip install` actually installs.


In [ ]:
if HAS_HATCH:
    result = subprocess.run(['hatch', 'build'], capture_output=True, text=True, cwd='./hatch_demo')
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    
    import os
    for f in sorted(os.listdir('./hatch_demo/dist')):
        size = os.path.getsize(f'./hatch_demo/dist/{f}')
        print(f'  {f}  ({size} bytes)')
    
    shutil.rmtree('./hatch_demo')
    print()
    print('Cleaned up demo project.')
else:
    print('Expected output of `hatch build`:')
    print()
    print('  Building source distribution...')
    print('  Building wheel...')
    print('  dist/cfd_post-0.1.0.tar.gz')
    print('  dist/cfd_post-0.1.0-py3-none-any.whl')
    print()
    print('The .whl file is a zip archive containing your package.')
    print('You can inspect it: unzip -l dist/cfd_post-0.1.0-py3-none-any.whl')

## 4. Install and Test

**Concept**: `hatch install` (or `pip install dist/*.whl`) installs the package. You can then `import` it from anywhere.


In [ ]:
if HAS_HATCH:
    print('To install the built wheel:')
    print()
    print('  pip install dist/cfd_post-0.1.0-py3-none-any.whl')
    print()
    print('Or in editable mode (for development):')
    print()
    print('  hatch install -e .')
    print()
    print('Then test:')
    print()
    print('  python -c "import cfd_post; print(cfd_post.__version__)"')
else:
    print('Expected output:')
    print()
    print('  $ pip install dist/cfd_post-0.1.0-py3-none-any.whl')
    print('  Processing ./dist/cfd_post-0.1.0-py3-none-any.whl')
    print('  Installing collected packages: cfd-post')
    print('  Successfully installed cfd-post-0.1.0')
    print()
    print('  $ python -c "import cfd_post; print(cfd_post.__version__)"')
    print('  0.1.0')

## 5. Hatch vs setuptools vs poetry

**Concept**: All three can build Python packages. The differences are in speed, configuration style, and ecosystem.


In [ ]:
comparison = (
    "                    setuptools    poetry    hatch\n"
    "                  ----------    ------    -----\n"
    "  Config file       setup.py      pyproject.toml  pyproject.toml\n"
    "  Speed             slow          medium    fast\n"
    "  Lock file         no            yes       yes (via uv)\n"
    "  Python mgmt       no            yes       no (use uv)\n"
    "  Build backend     setuptools    poetry-core hatchling\n"
    "  Ecosystem         huge          medium    growing\n"
    "  Learning curve    steep         medium    low\n"
)
print(comparison)
print()
print('Recommendation:')
print('  - New project: hatch + uv (fast, modern, simple)')
print('  - Existing project: keep setuptools, migrate when convenient')
print('  - Poetry: good if you want everything in one tool')

## Exercises

1. **Build a real package**: Take one of your CFD post-processing scripts. Wrap it in a `hatch` package. Build the wheel. Install it. Import it from a different directory.

2. **Versioning**: Change the version in `pyproject.toml` to `0.2.0`. Rebuild. Install the new version. Does `pip show cfd-post` show the new version?

3. **Dependencies**: Add `numpy` and `scipy` as dependencies. Build. Install in a fresh venv. Are the dependencies installed automatically?

4. **Publish to PyPI**: Create a test account on [test.pypi.org](https://test.pypi.org). Use `hatch publish` to upload your package. Install it from test.pypi.org on a different machine.

5. **vs uv**: Create the same project with `hatch` and `uv`. Compare: build time, configuration complexity, and developer experience. Which do you prefer?
